# 📋 Day 5: Assignment — Model Card + Manager Recommendation (Churn × Value)

## Overview

You will deliver a manager-ready artifact based on the Telco churn dataset:

1. A **churn classifier** ($p(\text{churn})$)
2. A **MonthlyCharges regressor** ($\widehat{\text{MonthlyCharges}}$)
3. A **Revenue-at-Risk** targeting list
4. A short **manager memo** with a deployment/monitoring plan

You must also include your **Independent Lab extension track** (A/B/C/D).

---
## Part 0: Setup

In [ ]:
!pip install -q -U pandas numpy scikit-learn shap google-genai
# Optional (Track C): !pip install -q -U flaml

## 🔧 GenAI Copilot Setup

In [ ]:
import os
import json
import warnings
from datetime import datetime, timezone

import google.genai as genai

# ── GenAI API Setup ────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Paste your GEMINI_API_KEY: ")
    os.environ["GEMINI_API_KEY"] = API_KEY

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-2.5-flash-lite"

# ── Logging Infrastructure ────────────────────────────────────
PROMPT_LOG = []

def _now():
    """Return current UTC timestamp in ISO format."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def log_interaction(role, content, label=None):
    """Log a GenAI interaction (prompt or response)."""
    entry = {
        "ts": _now(),
        "role": role,
        "content": content if isinstance(content, str) else json.dumps(content),
        "label": label or "",
    }
    PROMPT_LOG.append(entry)
    return entry

print("✅ GenAI + logging ready.")

## 🧾 Required: GenAI Copilot Log

Include **at least 3** unique uses of GenAI while working on this assignment:

- **Use #1:** …
  - Prompt summary: …
  - Change I made: …
  - What I verified manually: …

- **Use #2:** …
  - Prompt summary: …
  - Change I made: …
  - What I verified manually: …

- **Use #3:** …
  - Prompt summary: …
  - Change I made: …
  - What I verified manually: …

## ✅ Verification Checklist (required)

- [ ] Metrics computed on a **holdout test set**
- [ ] Preprocessing inside the **pipeline** (no train/test contamination)
- [ ] Basic **leakage scan** completed and documented
- [ ] SHAP plots generated from the final model (global + local)
- [ ] Manager memo references **actual metrics/plots** from this notebook
- [ ] Required artifacts saved (CSV, JSON, memo)
- [ ] GenAI copilot log includes **at least 3 substantive uses**

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, accuracy_score,
    mean_absolute_error, mean_squared_error, r2_score
)

import shap
shap.initjs()

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

print("✅ All imports ready.")

---
## Part 1: Data, leakage checks, and split (Deliverable 1)

✅ **Deliverable:** document what you removed/changed and why.

In [ ]:
url = "https://raw.githubusercontent.com/blastchar/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip().replace("", np.nan), errors="coerce")
customer_ids = df["customerID"].copy()

y_clf = (df["Churn"] == "Yes").astype(int)
y_reg = df["MonthlyCharges"].astype(float)
X = df.drop(columns=["customerID", "Churn"])

print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Churn rate: {y_clf.mean():.1%}")

### ── TODO: Leakage check notes ───────────────────────

Write a few sentences:
- What would count as leakage in a churn setting?
- Which columns did you exclude (if any) and why?

In [ ]:
# TODO: Complete leakage analysis
# Example considerations:
# - Do NOT use 'Churn' itself as a feature
# - Avoid features only known AFTER churn occurs (e.g., end date)
# - Be careful with variables that might change during preprocessing
# - Check for data contamination between train/test (handled by pipeline)

leakage_notes = """
TODO: Document leakage checks:
1. Columns excluded: ...
2. Reason: ...
3. Pipeline ensures no train/test contamination: ...
"""

print(leakage_notes)

In [ ]:
# ── Train/test split ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_clf
)
yreg_train = y_reg.loc[X_train.index]
yreg_test = y_reg.loc[X_test.index]

print(f"Train: {X_train.shape[0]:,} rows (churn rate: {y_train.mean():.1%})")
print(f"Test:  {X_test.shape[0]:,} rows (churn rate: {y_test.mean():.1%})")

---
## Part 2: Model comparison — churn classification (Deliverable 2)

✅ **Required:** logistic regression, random forest, gradient boosting.

In [ ]:
# ── Preprocessing Pipeline ────────────────────────────
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

# Handle sklearn API variations
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)]), categorical_cols),
    ]
)

print(f"✅ Preprocessing pipeline ready.")
print(f"   Numeric columns: {len(numeric_cols)}")
print(f"   Categorical columns: {len(categorical_cols)}")

In [ ]:
# ── Helper functions ──────────────────────────────────

def evaluate_classifier(name, model, X_te, y_te, threshold=0.5):
    """Evaluate a classifier and return metrics + probabilities."""
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    out = {
        "model": name,
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
    }
    return out, proba, pred


def lift_by_decile(y_true, y_score, n_bins=10):
    """Compute lift by decile (manager-friendly ranking metric)."""
    tmp = pd.DataFrame({"y": y_true, "score": y_score}).copy()
    tmp["decile"] = pd.qcut(tmp["score"].rank(method="first"), q=n_bins, labels=False) + 1
    overall = tmp["y"].mean()
    table = (
        tmp.groupby("decile")
           .agg(n=("y", "size"), churn_rate=("y", "mean"), avg_score=("score", "mean"))
           .sort_index(ascending=False)
           .reset_index()
    )
    table["lift"] = table["churn_rate"] / overall
    return table, overall


def eval_regression(name, model, X_te, y_te):
    """Evaluate a regression model."""
    pred = model.predict(X_te)
    out = {
        "model": name,
        "mae": mean_absolute_error(y_te, pred),
        "rmse": mean_squared_error(y_te, pred, squared=False),
        "r2": r2_score(y_te, pred)
    }
    return out, pred

print("✅ Helper functions defined.")

In [ ]:
# ── Train three classification models ──────────────────
logit = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=600, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced_subsample"))])
gb = Pipeline([("prep", preprocess), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])

for m in [logit, rf, gb]:
    m.fit(X_train, y_train)

m1, p1, pred1 = evaluate_classifier("logit", logit, X_test, y_test)
m2, p2, pred2 = evaluate_classifier("rf", rf, X_test, y_test)
m3, p3, pred3 = evaluate_classifier("gb", gb, X_test, y_test)

clf_results = pd.DataFrame([m1, m2, m3]).sort_values(["pr_auc", "roc_auc"], ascending=False)

print("\n📊 Classification Model Results:")
print(clf_results.to_string(index=False))

### ── TODO: Choose your final churn model ──────────────

Explain briefly:
- which model you chose and why (metric + interpretability)
- which threshold rule you will use and why

In [ ]:
# Pick the best model by PR-AUC
final_clf_name = clf_results.iloc[0]["model"]
final_clf = {"logit": logit, "rf": rf, "gb": gb}[final_clf_name]
p_final = {"logit": p1, "rf": p2, "gb": p3}[final_clf_name]

print(f"Selected model: {final_clf_name}")
print(f"\nModel justification:")
print(f"  PR-AUC: {clf_results.iloc[0]['pr_auc']:.3f} (best among three models)")
print(f"  ROC-AUC: {clf_results.iloc[0]['roc_auc']:.3f}")
print(f"\nThreshold rule (TODO): Describe your threshold decision...")

### ── GenAI: Interpret Model Comparison ────────────────

In [ ]:
# ── GenAI: Generate model interpretation ────────────────
metrics_summary = clf_results.to_string(index=False)

interpret_prompt = f"""Here are churn classification model results on a holdout test set:

{metrics_summary}

Context: We are building a churn retention campaign where the cost of missing a churner (false negative)
is roughly 5× the cost of contacting a non-churner (false positive).

In 3–4 sentences, recommend which model to use and why. Explain the precision-recall tradeoff and
how it impacts our retention targeting strategy."""

log_interaction("user", interpret_prompt, label="model_interpretation")

response = client.models.generate_content(model=MODEL_ID, contents=interpret_prompt)
interpretation = response.text
log_interaction("assistant", interpretation, label="model_interpretation")

print("🤖 GenAI Model Interpretation:")
print("=" * 70)
print(interpretation)
print()
print("TODO: Review the interpretation above and note any changes you made below.")

---
## Part 3: Model comparison — MonthlyCharges regression (Deliverable 2)

✅ **Required:** ridge, random forest regressor, gradient boosting regressor.

In [ ]:
# ── Regression features (exclude MonthlyCharges itself) ───
Xr_train = X_train.drop(columns=["MonthlyCharges"])
Xr_test = X_test.drop(columns=["MonthlyCharges"])

num_r = Xr_train.select_dtypes(include=["number"]).columns.tolist()
cat_r = Xr_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocess_r = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_r),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)]), cat_r),
    ]
)

ridge = Pipeline([("prep", preprocess_r), ("reg", Ridge(alpha=1.0))])
rfr = Pipeline([("prep", preprocess_r), ("reg", RandomForestRegressor(n_estimators=600, n_jobs=-1, random_state=RANDOM_STATE))])
gbr = Pipeline([("prep", preprocess_r), ("reg", GradientBoostingRegressor(random_state=RANDOM_STATE))])

for m in [ridge, rfr, gbr]:
    m.fit(Xr_train, yreg_train)

r1, v1 = eval_regression("ridge", ridge, Xr_test, yreg_test)
r2m, v2 = eval_regression("rfr", rfr, Xr_test, yreg_test)
r3, v3 = eval_regression("gbr", gbr, Xr_test, yreg_test)

reg_results = pd.DataFrame([r1, r2m, r3]).sort_values(["mae", "rmse"], ascending=True)

print("\n📊 Regression Model Results:")
print(reg_results.to_string(index=False))

---
## Part 4: Explainability with SHAP (Deliverable 4)

✅ **Required:** 1 global plot + 1 local explanation.

In [ ]:
# ── SHAP Setup ────────────────────────────────────────────
prep_fitted = final_clf.named_steps["prep"]
clf_fitted = final_clf.named_steps["clf"]
X_train_enc = prep_fitted.transform(X_train)
feature_names = prep_fitted.get_feature_names_out()

# Sample for SHAP speed
sample_idx = np.random.RandomState(RANDOM_STATE).choice(X_train_enc.shape[0], size=min(800, X_train_enc.shape[0]), replace=False)
X_shap = X_train_enc[sample_idx]

# Create explainer with fallback
explainer = None
try:
    explainer = shap.TreeExplainer(clf_fitted)
    shap_values = explainer.shap_values(X_shap)
except Exception as e:
    print(f"TreeExplainer failed, falling back: {e}")
    explainer = shap.Explainer(clf_fitted, X_shap)
    shap_values = explainer(X_shap)

print(f"✅ SHAP explainer ready. Shap values shape: {np.array(shap_values).shape}")

In [ ]:
# ── SHAP Global Summary Plot ─────────────────────────────
plt.figure(figsize=(10, 6))
try:
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_shap, feature_names=feature_names, max_display=15, show=False)
    else:
        shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=15, show=False)
    plt.title(f"SHAP Summary Plot — {final_clf_name}")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not render summary plot: {e}")

In [ ]:
# ── SHAP Local Explanation (one customer) ────────────────
row_i = 0
x_one = X_shap[row_i:row_i+1]

try:
    if hasattr(explainer, "__call__") and not isinstance(shap_values, list):
        exp = explainer(x_one)
        shap.plots.waterfall(exp[0])
    else:
        # TreeExplainer legacy API
        sv = shap_values[1][row_i] if isinstance(shap_values, list) else shap_values[row_i]
        base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        shap.plots._waterfall.waterfall_legacy(base, sv, feature_names=feature_names)
except Exception as e:
    print(f"Local explanation failed: {e}")

### ── GenAI: SHAP Narrative for Individual Customer ────────

In [ ]:
# ── GenAI: Generate customer narrative from SHAP ──────────
if isinstance(shap_values, list):
    sv_row = shap_values[1][row_i]
else:
    sv_row = shap_values[row_i]

shap_df = pd.DataFrame({"feature": feature_names, "shap_value": sv_row})
shap_df["abs_shap"] = shap_df["shap_value"].abs()
top_features = shap_df.nlargest(6, "abs_shap")

shap_summary = "\n".join(
    f"  - {row['feature']}: SHAP={row['shap_value']:+.3f} ({'increases' if row['shap_value'] > 0 else 'decreases'} churn risk)"
    for _, row in top_features.iterrows()
)

narrative_prompt = f"""A customer has been flagged by our churn prediction model.
The baseline churn rate is {y_test.mean():.0%}.

Top factors driving this prediction (SHAP values):
{shap_summary}

Write a 3-sentence explanation for a retention manager who needs to decide
whether to call this customer. Use plain business language, not technical jargon."""

log_interaction("user", narrative_prompt, label="shap_narrative")

response = client.models.generate_content(model=MODEL_ID, contents=narrative_prompt)
narrative = response.text
log_interaction("assistant", narrative, label="shap_narrative")

print("🤖 GenAI Narrative for Retention Manager:")
print("=" * 70)
print(narrative)
print()
print("TODO: Review the narrative and note any changes below.")

---
## Part 5: Revenue-at-Risk call list (Deliverable 5)

In [ ]:
# ── Pick final regression model ───────────────────────────
final_reg_name = reg_results.iloc[0]["model"]
final_reg = {"ridge": ridge, "rfr": rfr, "gbr": gbr}[final_reg_name]
v_final = final_reg.predict(Xr_test)
v_final = np.clip(v_final, 0, None)

# ── Compute Revenue-at-Risk ──────────────────────────────
call_list = pd.DataFrame({
    "customerID": customer_ids.loc[X_test.index].values,
    "p_churn": p_final,
    "pred_monthly_charges": v_final,
})
call_list["revenue_at_risk"] = call_list["p_churn"] * call_list["pred_monthly_charges"]

call_list = call_list.sort_values("revenue_at_risk", ascending=False)

print(f"\n📊 Top 10 Customers by Revenue-at-Risk:")
print(call_list.head(10).to_string(index=False))
print(f"\nTotal Revenue-at-Risk (test set): ${call_list['revenue_at_risk'].sum():,.0f}")

### ── TODO: Add simple reason codes (top SHAP drivers) ────

You can do this approximately:
- take the top 2–3 absolute SHAP features for each selected customer
- store them as strings (e.g., `"Contract=Month-to-month; tenure(low); TechSupport=No"`)

Tip: For grading, it's fine to compute reason codes for the **top N** only.

In [ ]:
# TODO: Compute SHAP-based reason codes
# For the top N customers in the call list, extract their top 2-3 SHAP drivers
# and create a simple text summary.

# Placeholder code structure:
# reason_codes = []
# for idx in call_list.head(100).index:
#     # Get SHAP values for this customer
#     shap_row = shap_values[1][idx] if isinstance(shap_values, list) else shap_values[idx]
#     # Extract top 3 features
#     top_k = 3
#     top_idx = np.argsort(np.abs(shap_row))[-top_k:][::-1]
#     reasons = "; ".join([feature_names[i] for i in top_idx])
#     reason_codes.append(reasons)
#
# call_list['reason_codes'] = reason_codes

print("TODO: Add reason_codes column to call_list")

### ── GenAI: Generate Action Recommendations ────────────

In [ ]:
# ── GenAI: Action recommendations for top-at-risk ────────
top_10_stats = call_list.head(10)[["p_churn", "pred_monthly_charges", "revenue_at_risk"]].describe().to_string()

action_prompt = f"""Based on our revenue-at-risk analysis for the top 10 at-risk customers, recommend 3 concrete business actions.

Top 10 customer statistics:
{top_10_stats}

Consider retention strategy, offer design, and outreach timing. Keep it practical."""

log_interaction("user", action_prompt, label="action_recommendations")

response = client.models.generate_content(model=MODEL_ID, contents=action_prompt)
actions = response.text
log_interaction("assistant", actions, label="action_recommendations")

print("🤖 GenAI Action Recommendations:")
print("=" * 70)
print(actions)
print()
print("TODO: Review recommendations and incorporate into your manager memo.")

---
## Part 6: Save Artifacts

In [ ]:
# ── Save call list ───────────────────────────────────────
call_list_out = "day5_assignment_call_list.csv"
call_list.to_csv(call_list_out, index=False)
print(f"✅ Saved call list to {call_list_out}")

# ── Save metrics ─────────────────────────────────────────
metrics = {
    "final_churn_model": final_clf_name,
    "final_value_model": final_reg_name,
    "churn_metrics": clf_results.to_dict(orient="records"),
    "value_metrics": reg_results.to_dict(orient="records"),
    "extension_track": "TODO: A/B/C/D",
    "threshold_rule": "TODO: describe threshold/top-N rule",
    "verification": {
        "holdout_test": True,
        "pipeline_no_leakage": True,
        "leakage_scan_done": False,  # TODO: set to True when complete
        "memo_references_real_outputs": False,  # TODO: set to True when memo is done
    },
}

metrics_out = "day5_assignment_metrics.json"
with open(metrics_out, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_out}")

---
## Part 7: Manager Memo (Deliverable 3)

✅ Create a 1-page memo as a separate markdown file.

### ── GenAI: Draft Manager Memo ────────────────────────

In [ ]:
# ── GenAI: Generate manager memo ──────────────────────────
memo_context = f"""Please write a 1-page executive memo for the VP of Customer Success on our churn targeting initiative.

Key facts:
- Best churn model: {final_clf_name} (PR-AUC: {clf_results.iloc[0]['pr_auc']:.3f}, ROC-AUC: {clf_results.iloc[0]['roc_auc']:.3f})
- Best value model: {final_reg_name} (MAE: ${reg_results.iloc[0]['mae']:.2f})
- Total revenue at risk: ${call_list['revenue_at_risk'].sum():,.0f}
- Test set size: {len(X_test)} customers
- Baseline churn rate: {y_test.mean():.1%}

The memo should include:
1. Executive summary (2-3 sentences on business impact)
2. Model performance (key metrics in plain language)
3. Targeting strategy (how to prioritize customers)
4. Top risk drivers (3-4 main churn factors from SHAP)
5. Recommended actions (3-4 next steps)
6. Implementation risks and monitoring plan

Keep it professional, actionable, and free of technical jargon."""

log_interaction("user", memo_context, label="manager_memo")

response = client.models.generate_content(model=MODEL_ID, contents=memo_context)
memo_draft = response.text
log_interaction("assistant", memo_draft, label="manager_memo")

print("🤖 GenAI Draft Manager Memo:")
print("=" * 70)
print(memo_draft)
print()
print("TODO: Review, edit, and save the final memo below.")

In [ ]:
# ── Save manager memo ────────────────────────────────────
# TODO: Copy the final memo text from the GenAI draft above, make edits, paste below

memo = """# TODO: Day 5 Manager Memo — Churn × Value Targeting

[Copy the final memo from the GenAI draft above and make any edits here.]

## Executive summary
- TODO

## What decision are we supporting?
- TODO

## Model performance
- Churn model: TODO (metric + interpretation)
- Value model: TODO (MAE, business meaning)

## Targeting plan
- Capacity: TODO
- Rule: TODO (top-N by revenue-at-risk or threshold)
- Expected impact: TODO

## Top drivers (from SHAP)
- TODO (3–5 drivers + business meaning)

## Risks & monitoring
- Data drift: TODO
- Segment risk: TODO
- Calibration drift: TODO

## Next steps
- TODO (A/B test, rollout plan, etc.)
"""

memo_path = "day5_assignment_manager_memo.md"
with open(memo_path, "w") as f:
    f.write(memo)
print(f"✅ Saved manager memo to {memo_path}")

---
## Part 8: GenAI Copilot Log Summary

In [ ]:
# ── Export Copilot Log ───────────────────────────────────
print(f"📋 Copilot interactions logged: {len(PROMPT_LOG)}")
print()

for i, entry in enumerate(PROMPT_LOG, 1):
    print(f"--- Log Entry {i} ---")
    print(f"Time: {entry['ts']}")
    print(f"Role: {entry['role']}")
    print(f"Label: {entry['label']}")
    if len(entry['content']) > 200:
        print(f"Content (first 200 chars): {entry['content'][:200]}...")
    else:
        print(f"Content: {entry['content']}")
    print()

# Save log as JSON
log_path = "day5_assignment_copilot_log.json"
with open(log_path, "w") as f:
    json.dump(PROMPT_LOG, f, indent=2)
print(f"✅ Saved copilot log to {log_path}")

---
## ✅ Final Verification Checklist

Before submission, verify all items are complete:

- [ ] Metrics computed on a **holdout test set** (25% split)
- [ ] Preprocessing inside a **pipeline** (no train/test leakage)
- [ ] **Leakage scan** documented in Part 1
- [ ] **Three churn models** trained (logit, RF, GB) with metrics
- [ ] **Three regression models** trained (ridge, RF, GB) with MAE/RMSE/R2
- [ ] **SHAP summary plot** (global feature importance)
- [ ] **SHAP waterfall** for at least 1 customer (local explanation)
- [ ] **Revenue-at-Risk** ranked call list created and saved as CSV
- [ ] **Manager memo** saved as markdown file, with real metrics referenced
- [ ] **Metrics JSON** saved with model names, scores, threshold rule, extension track
- [ ] **Copilot log** includes at least 3 GenAI uses with clear labels (model interpretation, SHAP narrative, action recommendations, memo draft)
- [ ] **Extension track** from Independent Lab documented in metrics JSON
- [ ] All output files saved to working directory
- [ ] Notebook runs end-to-end without errors

### Files to submit:
1. `Day5_Assignment5.ipynb` (this notebook)
2. `day5_assignment_call_list.csv` (ranked customers)
3. `day5_assignment_metrics.json` (model scores + metadata)
4. `day5_assignment_manager_memo.md` (executive summary)
5. `day5_assignment_copilot_log.json` (GenAI interaction log)